# 🌉 IC-SHM 2026 Project 2 — Multi-view Semantic 3D Visualization

This notebook visualizes the semantically-labeled 3D point cloud of the bridge structure generated from COLMAP SfM parameters and Ground-Truth 2D semantic masks.

### 🏷️ Semantic Classes & Color Legend:
- `0: background` — Gray `[128, 128, 128]`
- `1: deck` — Red `[255, 0, 0]`
- `2: stay_cable` — Cyan `[0, 255, 255]`
- `3: tower` — Green `[0, 255, 0]`
- `4: foundation` — Yellow `[255, 255, 0]`

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.reconstruction.visualizer import read_ply_file, create_interactive_3d_figure, CLASS_NAMES, CLASS_HEX_COLORS

## 1. Load PLY Point Cloud Data

In [3]:
PLY_PATH = '../outputs/point_clouds/semantic_bridge_sparse.ply'

xyz, rgb, cids = read_ply_file(PLY_PATH)
print(f"✅ Loaded {len(xyz):,} 3D points from '{PLY_PATH}'")

# Class distribution table
unique, counts = np.unique(cids, return_counts=True)
df_dist = pd.DataFrame({
    'Class ID': unique,
    'Class Name': [CLASS_NAMES[c] for c in unique],
    'Point Count': counts,
    'Percentage (%)': (counts / len(cids) * 100).round(2)
})
display(df_dist)

# Donut Chart of Semantic Classes
fig_pie = px.pie(
    df_dist, values='Point Count', names='Class Name',
    title='3D Point Cloud Semantic Class Distribution',
    color='Class Name',
    color_discrete_map={CLASS_NAMES[c]: CLASS_HEX_COLORS[c] for c in unique},
    hole=0.4
)
fig_pie.update_layout(template='plotly_dark')
fig_pie.show()

✅ Loaded 86,336 3D points from '../outputs/point_clouds/semantic_bridge_sparse.ply'


,Class ID,Class Name,Point Count,Percentage (%)
0,0,background,60761,70.38
1,1,deck,11515,13.34
2,2,stay_cable,8448,9.79
3,3,tower,3350,3.88
4,4,foundation,2262,2.62


## 2. Interactive 3D Point Cloud Rendering

Use your mouse to rotate (left click), pan (right click), or zoom (scroll) the 3D model. Click legend items to show/hide specific structural components.

In [4]:
fig_3d = create_interactive_3d_figure(xyz, rgb, cids, point_size=2.0)
fig_3d.show()

## 3. Structural Component Isolation (`stay_cable` + `tower` + `foundation` + `deck`)

Filter out the background to focus exclusively on the bridge structural elements.

In [ ]:
# Mask out background (class 0)
struct_mask = (cids != 0)
xyz_struct = xyz[struct_mask]
rgb_struct = rgb[struct_mask]
cids_struct = cids[struct_mask]

print(f"🔍 Isolated {len(xyz_struct):,} bridge structural 3D points (excluding background)")
fig_struct = create_interactive_3d_figure(xyz_struct, rgb_struct, cids_struct, point_size=2.5)
fig_struct.update_layout(title="🌉 Isolated Bridge Structural Components (Deck, Cables, Tower, Foundation)")
fig_struct.show()